# Feature Engineering — NYC Yellow Taxi

En este notebook se crearán variables analíticas derivadas del dataset procesado de Yellow Taxi correspondiente a julio de 2026.

El objetivo es transformar las variables originales en características útiles para el análisis de movilidad urbana, demanda, comportamiento temporal, eficiencia de los viajes y análisis geográfico.

El procesamiento se realizará por bloques para mantener un uso eficiente de memoria.

In [1]:
         # Importar librerías
from pathlib import Path
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import gc

# Configurar formato numérico para facilitar la lectura
pd.options.display.float_format = "{:,.2f}".format

# Definir rutas
ruta_clean = Path(
    "../data/processed/yellow_tripdata_2026-07_clean.parquet"
)

ruta_features = Path(
    "../data/processed/yellow_tripdata_2026-07_features.parquet"
)

# Abrir archivo procesado
archivo_clean = pq.ParquetFile(ruta_clean)

print("Archivo encontrado:", ruta_clean.exists())
print("Filas:", archivo_clean.metadata.num_rows)
print("Columnas:", archivo_clean.metadata.num_columns)
print("Row Groups:", archivo_clean.metadata.num_row_groups)

Archivo encontrado: True
Filas: 3530109
Columnas: 28
Row Groups: 4


In [2]:
# Crear variables temporales para analizar patrones de demanda
df = archivo_clean.read_row_group(0).to_pandas()

df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
df["pickup_day"] = df["tpep_pickup_datetime"].dt.day_name()
df["pickup_day_num"] = df["tpep_pickup_datetime"].dt.dayofweek
df["is_weekend"] = df["pickup_day_num"].isin([5, 6])

df[
    [
        "tpep_pickup_datetime",
        "pickup_hour",
        "pickup_day",
        "pickup_day_num",
        "is_weekend"
    ]
].head(10)

,tpep_pickup_datetime,pickup_hour,pickup_day,pickup_day_num,is_weekend
0,2026-07-01 00:36:25,0,Wednesday,2,False
1,2026-07-01 00:02:53,0,Wednesday,2,False
2,2026-07-01 00:18:56,0,Wednesday,2,False
3,2026-07-01 00:35:06,0,Wednesday,2,False
4,2026-07-01 00:36:59,0,Wednesday,2,False
5,2026-07-01 00:00:37,0,Wednesday,2,False
6,2026-07-01 00:33:53,0,Wednesday,2,False
7,2026-07-01 00:52:53,0,Wednesday,2,False
8,2026-07-01 00:11:39,0,Wednesday,2,False
9,2026-07-01 00:11:54,0,Wednesday,2,False


In [3]:
# Crear franjas horarias para analizar patrones de demanda
condiciones = [
    df["pickup_hour"].between(5, 11),
    df["pickup_hour"].between(12, 16),
    df["pickup_hour"].between(17, 21)
]

categorias = [
    "Morning",
    "Afternoon",
    "Evening"
]

df["time_period"] = np.select(
    condiciones,
    categorias,
    default="Night"
)

df[
    ["pickup_hour", "time_period"]
].value_counts().sort_index()

pickup_hour  time_period
0            Night          26981
1            Night          16796
2            Night          10939
3            Night           7461
4            Night           5447
5            Morning         6917
6            Morning        14016
7            Morning        26174
8            Morning        38052
9            Morning        45417
10           Morning        49888
11           Morning        55218
12           Afternoon      60608
13           Afternoon      64783
14           Afternoon      68312
15           Afternoon      69343
16           Afternoon      73220
17           Evening        76734
18           Evening        75213
19           Evening        62588
20           Evening        54751
21           Evening        54797
22           Night          48340
23           Night          36581
Name: count, dtype: int64

In [4]:
# Velocidad promedio de viaje
df["avg_speed_mph"] = np.where(
    df["trip_duration_min"] > 0,
    df["trip_distance"] / (df["trip_duration_min"] / 60),
    np.nan
)

df["avg_speed_mph"].describe()

count   1,031,824.00
mean           11.13
std            39.91
min             0.00
25%             6.68
50%             9.17
75%            12.83
max        28,854.00
Name: avg_speed_mph, dtype: float64

In [5]:
# Crear una flag para identificar velocidades promedio superiores a 60 mph
df["flag_speed_extreme"] = df["avg_speed_mph"] > 60

print(
    "Viajes con velocidad > 60 mph:",
    df["flag_speed_extreme"].sum()
)

print(
    "Porcentaje:",
    round(df["flag_speed_extreme"].mean() * 100, 3),
    "%"
)

Viajes con velocidad > 60 mph: 260
Porcentaje: 0.025 %


In [6]:
# Creamos una nueva variable de costo por milla recorrida
df["cost_per_mile"] = np.where(
    df["trip_distance"] > 0,
    df["total_amount"] / df["trip_distance"],
    np.nan
)

df["cost_per_mile"].describe()


count   1,033,116.00
mean           21.59
std           222.21
min       -16,675.00
25%             8.70
50%            12.38
75%            17.37
max        50,000.00
Name: cost_per_mile, dtype: float64

In [7]:
# Calculamos el monto antes de las propinas 
df["amount_before_tip"] = (
    df["total_amount"] - df["tip_amount"]
)

df["tip_percentage"] = np.where(
    df["amount_before_tip"] > 0,
    (df["tip_amount"] / df["amount_before_tip"]) * 100,
    np.nan
)

df["tip_percentage"].describe()

count   1,041,997.00
mean           14.96
std            13.61
min             0.00
25%             7.05
50%            20.00
75%            20.00
max         3,428.57
Name: tip_percentage, dtype: float64

In [8]:
# Crear categorías de distancia
condiciones_distancia = [
    df["trip_distance"] == 0,
    df["trip_distance"].between(0, 2, inclusive="right"),
    df["trip_distance"].between(2, 5, inclusive="right"),
    df["trip_distance"].between(5, 10, inclusive="right"),
    df["trip_distance"] > 10
]

categorias_distancia = [

    "Zero",
    "Short",
    "Medium",
    "Long",
    "Very Long"
]

df["distance_category"] = np.select(
    condiciones_distancia,
    categorias_distancia,
    default="Unknown"
)

df["distance_category"].value_counts()

distance_category
Short        604354
Medium       252915
Long          90454
Very Long     85393
Zero          15460
Name: count, dtype: int64

### Categorías de distancia

Para facilitar el análisis, los viajes se clasifican según la distancia recorrida:

| **Categoría** | **Distancia** |
| ------------- | ------------- |
| `Zero`        | 0 millas      |
| `Short`       | >0 hasta 2    |
| `Medium`      | >2 hasta 5    |
| `Long`        | >5 hasta 10   |
| `Very Long`   | >10           |

In [9]:
#Creamos categorias de duracion
condiciones_duracion = [
    df["trip_duration_min"] == 0,
    df["trip_duration_min"].between(0, 5, inclusive="right"),
    df["trip_duration_min"].between(5, 15, inclusive="right"),
    df["trip_duration_min"].between(15, 30, inclusive="right"),
    df["trip_duration_min"].between(30, 60, inclusive="right"),
    df["trip_duration_min"] > 60
]

categorias_duracion = [
    "Zero",
    "Very Short",
    "Short",
    "Medium",
    "Long",
    "Very Long"
]

df["duration_category"] = np.select(
    condiciones_duracion,
    categorias_duracion,
    default="Unknown"
)

df["duration_category"].value_counts()

duration_category
Short         494429
Medium        293170
Very Short    129054
Long          101661
Zero           16752
Very Long      13510
Name: count, dtype: int64

### Categorías de duración

Para facilitar el análisis, los viajes se clasifican según su duración:

| **Categoría** | **Duración** |
| ------------- | ------------ |
| `Zero`        | 0 minutos    |
| `Very Short`  | >0 hasta 5   |
| `Short`       | >5 hasta 15  |
| `Medium`      | >15 hasta 30 |
| `Long`        | >30 hasta 60 |
| `Very Long`   | >60          |


In [10]:
# Revisamos todas las variables nuevas
features_nuevas = [
    "pickup_hour",
    "pickup_day",
    "pickup_day_num",
    "is_weekend",
    "time_period",
    "avg_speed_mph",
    "flag_speed_extreme",
    "cost_per_mile",
    "amount_before_tip",
    "tip_percentage",
    "distance_category",
    "duration_category"
]

print("Features creadas:", len(features_nuevas))

df[features_nuevas].dtypes

Features creadas: 12


pickup_hour             int32
pickup_day             object
pickup_day_num          int32
is_weekend               bool
time_period            object
avg_speed_mph         float64
flag_speed_extreme       bool
cost_per_mile         float64
amount_before_tip     float64
tip_percentage        float64
distance_category      object
duration_category      object
dtype: object

In [11]:
# Pasamos todos los datos a string
def crear_features(df):
    df = df.copy()

    # Variables temporales
    df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
    df["pickup_day"] = df["tpep_pickup_datetime"].dt.day_name().astype("string")
    df["pickup_day_num"] = df["tpep_pickup_datetime"].dt.dayofweek

    df["is_weekend"] = df["pickup_day_num"].isin([5, 6])

    # Período del día
    condiciones = [
        df["pickup_hour"].between(5, 11),
        df["pickup_hour"].between(12, 16),
        df["pickup_hour"].between(17, 21)
    ]

    df["time_period"] = np.select(
        condiciones,
        ["Morning", "Afternoon", "Evening"],
        default="Night"
    )

    df["time_period"] = df["time_period"].astype("string")

    # Velocidad promedio
    df["avg_speed_mph"] = np.where(
        df["trip_duration_min"] > 0,
        df["trip_distance"] / (df["trip_duration_min"] / 60),
        np.nan
    )

    df["flag_speed_extreme"] = df["avg_speed_mph"] > 60

    # Costo por milla
    df["cost_per_mile"] = np.where(
        df["trip_distance"] > 0,
        df["total_amount"] / df["trip_distance"],
        np.nan
    )

    # Propina porcentual
    df["amount_before_tip"] = df["total_amount"] - df["tip_amount"]

    df["tip_percentage"] = np.where(
        df["amount_before_tip"] > 0,
        (df["tip_amount"] / df["amount_before_tip"]) * 100,
        np.nan
    )

    # Categoría de distancia
    condiciones_distancia = [
        df["trip_distance"] == 0,
        df["trip_distance"].between(0, 2, inclusive="right"),
        df["trip_distance"].between(2, 5, inclusive="right"),
        df["trip_distance"].between(5, 10, inclusive="right"),
        df["trip_distance"] > 10
    ]

    df["distance_category"] = np.select(
        condiciones_distancia,
        ["Zero", "Short", "Medium", "Long", "Very Long"],
        default="Unknown"
    ).astype("str")

    # Categoría de duración
    condiciones_duracion = [
        df["trip_duration_min"] == 0,
        df["trip_duration_min"].between(0, 5, inclusive="right"),
        df["trip_duration_min"].between(5, 15, inclusive="right"),
        df["trip_duration_min"].between(15, 30, inclusive="right"),
        df["trip_duration_min"].between(30, 60, inclusive="right"),
        df["trip_duration_min"] > 60
    ]

    df["duration_category"] = np.select(
        condiciones_duracion,
        ["Zero", "Very Short", "Short", "Medium", "Long", "Very Long"],
        default="Unknown"
    ).astype("str")

    return df

In [12]:
# Probar la función de creación de features en el primer bloque
bloque_prueba = archivo_clean.read_row_group(0).to_pandas()

bloque_features = crear_features(bloque_prueba)

print("Filas:", bloque_features.shape[0])
print("Columnas:", bloque_features.shape[1])

bloque_features[features_nuevas].head()

Filas: 1048576
Columnas: 40


,pickup_hour,pickup_day,pickup_day_num,is_weekend,time_period,avg_speed_mph,flag_speed_extreme,cost_per_mile,amount_before_tip,tip_percentage,distance_category,duration_category
0,0,Wednesday,2,False,Night,17.43,False,9.75,19.50,0.00,Short,Short
1,0,Wednesday,2,False,Night,13.41,False,6.67,21.35,0.00,Medium,Short
2,0,Wednesday,2,False,Night,21.93,False,7.57,19.25,24.99,Medium,Short
3,0,Wednesday,2,False,Night,26.37,False,7.40,61.71,19.93,Long,Medium
4,0,Wednesday,2,False,Night,NaN,False,5.98,47.95,20.00,Long,Zero


In [13]:
# Liberar memoria después de la prueba
del bloque_prueba
del bloque_features

gc.collect()

46

In [14]:
  # Eliminar el archivo de features anterior antes de generar una nueva versión
if ruta_features.exists():
    ruta_features.unlink()
    print("Archivo anterior eliminado.")
else:
    print("No existe un archivo anterior.")

Archivo anterior eliminado.


In [15]:
# Procesar todos los Row Groups y generar el dataset con features
writer = None

try:
    for i in range(archivo_clean.metadata.num_row_groups):

        print(
            f"Procesando Row Group {i + 1}/"
            f"{archivo_clean.metadata.num_row_groups}..."
        )

        # Leer un bloque
        bloque = archivo_clean.read_row_group(i).to_pandas()

        # Crear las features
        bloque_features = crear_features(bloque)

        # Convertir a PyArrow
        tabla = pa.Table.from_pandas(
            bloque_features,
            preserve_index=False
        )

        # Crear el archivo con el primer bloque
        if writer is None:
            writer = pq.ParquetWriter(
                ruta_features,
                tabla.schema,
                compression="snappy"
            )

        # Escribir el bloque
        writer.write_table(tabla)

        # Liberar memoria
        del bloque
        del bloque_features
        del tabla
        gc.collect()

        print(f"Row Group {i + 1} completado.")

finally:
    if writer is not None:
        writer.close()

print("Feature Engineering terminado.")

Procesando Row Group 1/4...
Row Group 1 completado.
Procesando Row Group 2/4...
Row Group 2 completado.
Procesando Row Group 3/4...
Row Group 3 completado.
Procesando Row Group 4/4...
Row Group 4 completado.
Feature Engineering terminado.


In [16]:
# Validar la estructura y el tamaño del dataset final con features
archivo_features = pq.ParquetFile(ruta_features)

print("Filas:", archivo_features.metadata.num_rows)
print("Columnas:", archivo_features.metadata.num_columns)
print("Row Groups:", archivo_features.metadata.num_row_groups)
print("Archivo existe:", ruta_features.exists())
print(
    "Tamaño MB:",
    round(ruta_features.stat().st_size / (1024 ** 2), 2)
)

Filas: 3530109
Columnas: 40
Row Groups: 4
Archivo existe: True
Tamaño MB: 135.73


In [17]:
# Resumen global de las principales features creadas
columnas_resumen = [
    "avg_speed_mph",
    "cost_per_mile",
    "tip_percentage",
    "flag_speed_extreme"
]

resumen_bloques = []

for i in range(archivo_features.metadata.num_row_groups):

    bloque = archivo_features.read_row_group(
        i,
        columns=columnas_resumen
    ).to_pandas()

    resumen_bloques.append(bloque)

    del bloque
    gc.collect()

resumen_features = pd.concat(
    resumen_bloques,
    ignore_index=True
)

resumen_features[
    ["avg_speed_mph", "cost_per_mile", "tip_percentage"]
].describe()

,avg_speed_mph,cost_per_mile,tip_percentage
count,"3,487,793.00","3,401,787.00","3,514,782.00"
mean,19.54,33.97,11.09
std,"2,662.99",312.04,15.27
min,0.00,"-20,100.00",0.00
25%,6.63,7.77,0.00
50%,9.32,11.44,11.43
75%,13.22,16.41,20.00
max,"1,853,626.95","129,427.00","9,999.00"


In [18]:
# Calcular la cantidad y el porcentaje de viajes con velocidad superior a 60 mph
cantidad_speed_extreme = resumen_features["flag_speed_extreme"].sum()

porcentaje_speed_extreme = (
    cantidad_speed_extreme / len(resumen_features)
) * 100

print("Velocidades > 60 mph:", cantidad_speed_extreme)
print("Porcentaje:", round(porcentaje_speed_extreme, 3), "%")

Velocidades > 60 mph: 1673
Porcentaje: 0.047 %


In [19]:
# Liberamos memoria
del resumen_features
del resumen_bloques
gc.collect()

0

## Conclusiones del Feature Engineering

Se procesaron **3.530.109 registros** mediante Row Groups para mantener un uso eficiente de memoria.

El dataset pasó de **28 a 40 variables**, incorporando 12 nuevas características analíticas.

### Variables creadas

**Variables temporales**
- `pickup_hour`: hora de inicio del viaje.
- `pickup_day`: día de la semana.
- `pickup_day_num`: número del día de la semana.
- `is_weekend`: identifica sábados y domingos.
- `time_period`: clasifica el viaje en Morning, Afternoon, Evening o Night.

**Variables de movilidad**
- `avg_speed_mph`: velocidad promedio estimada.
- `flag_speed_extreme`: identifica velocidades superiores a 60 mph.
- `distance_category`: clasifica los viajes según distancia.
- `duration_category`: clasifica los viajes según duración.

**Variables económicas**
- `cost_per_mile`: costo total por milla recorrida.
- `amount_before_tip`: monto total antes de la propina.
- `tip_percentage`: porcentaje de propina calculado.

### Hallazgos de calidad

Se identificaron **1.673 registros** con una velocidad promedio calculada superior a 60 mph, equivalentes aproximadamente al **0,047 %** del dataset.

Las variables `avg_speed_mph`, `cost_per_mile` y `tip_percentage` presentan valores extremos. Estos registros se conservaron para mantener la trazabilidad de los datos y permitir aplicar filtros específicos según el análisis.

Debido a estos valores extremos, métricas como la media deben interpretarse junto con medidas robustas como la mediana y los percentiles.

### Dataset generado

`data/processed/yellow_tripdata_2026-07_features.parquet`

- Filas: **3.530.109**
- Columnas: **40**
- Row Groups: **4**
- Tamaño aproximado: **135,73 MB**

Este dataset queda preparado para las siguientes etapas de análisis exploratorio, SQL y visualización.